In [8]:
# Setup (SparkSession + read data + build df_target):
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, row_number
from pyspark.sql.window import Window
import pandas as pd

# 1. Membuat SparkSession
spark = SparkSession.builder.appName("Tugas5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil dibuat. Versi Spark:", spark.version)

# 2. Membaca data transaksi dari HDFS
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/yan/tugas5/transaksi_tugas5.csv",
    header=True, inferSchema=True
)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))
print("df_transaksi berhasil dibaca. Jumlah baris:", df_transaksi.count())
df_transaksi.show(5)

# 3. Membuat df_target
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Arkan", "Azizi", "Audin", "Auzan", "Ahri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))
print("df_target berhasil dibuat. Jumlah baris:", df_target.count())
df_target.show()

SparkSession berhasil dibuat. Versi Spark: 3.5.9
df_transaksi berhasil dibaca. Jumlah baris: 500
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows

df_target berhasil dibuat. Jumlah baris: 5
+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------

In [9]:
# A. Join & Perbandingan Target
ringkasan_kota = df_transaksi.groupBy("kota").agg(spark_sum("pendapatan").alias("total_pendapatan"))

hasil_a = ringkasan_kota.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", col("total_pendapatan") / col("target_bulanan") * 100) \
    .orderBy(col("pencapaian_persen").desc())
hasil_a.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|      Ahri|152.16666666666669|
|      Solo|        33475000|      40000000|     Auzan|           83.6875|
|Yogyakarta|        47275000|      60000000|     Azizi| 78.79166666666667|
|  Magelang|        31650000|      45000000|     Arkan| 70.33333333333334|
|  Semarang|        38175000|      55000000|     Audin|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [10]:
# B. Window Function — Kategori Terlaris per Kota
ringkasan_kota_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)
window_b = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

hasil_b = ringkasan_kota_kategori.withColumn("rn", row_number().over(window_b)) \
    .filter(col("rn") == 1) \
    .drop("rn") \
    .orderBy("kota")
hasil_b.show()

+----------+--------------------+----------------+
|      kota|            kategori|total_pendapatan|
+----------+--------------------+----------------+
|  Magelang|Kesehatan & Kecan...|         7275000|
| Purworejo|Kesehatan & Kecan...|        10075000|
|  Semarang|        Rumah Tangga|        11125000|
|      Solo|Kesehatan & Kecan...|         8425000|
|Yogyakarta|             Fashion|        13325000|
+----------+--------------------+----------------+



In [11]:
# C. Spark SQL
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

hasil_c = spark.sql('''
    SELECT t.kota, tg.pic_cabang, COUNT(*) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')
hasil_c.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|      Ahri|             116|
|Yogyakarta|     Azizi|             110|
|      Solo|     Auzan|              95|
|  Semarang|     Audin|              93|
|  Magelang|     Arkan|              86|
+----------+----------+----------------+



# D. Kesimpulan
Cabang dengan kinerja terbaik adalah Purworejo. Pencapaiannya mencapai 152% dari target bulanan, artinya sudah melebihi target yang ditetapkan. Kategori "Kesehatan & Kecantikan" jadi penyumbang pendapatan terbesar di cabang ini. Sebaliknya, cabang yang paling perlu diperhatikan manajemen adalah Semarang, karena pencapaiannya paling rendah di antara semua cabang, hanya 69% dari target bulanan sebesar Rp55.000.000. Jika dilihat dari jumlah transaksinya, Semarang sebenarnya tidak jauh berbeda dari cabang lain, jadi masalahnya bukan karena kurangnya transaksi, melainkan nilai rata-rata tiap transaksi yang masih kecil. Manajemen bisa mempertimbangkan strategi untuk mendorong penjualan kategori terlaris di Semarang, yaitu "Rumah Tangga", misalnya lewat promo atau bundling produk, supaya pendapatan cabang ini bisa lebih mendekati target bulanannya.